In [2]:
import spacy
from typing import List
import wikipedia
from sentence_transformers import SentenceTransformer, util


In [4]:
nlp = spacy.load("en_core_web_sm")

sbert = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def extract_claims(text: str) -> List[str]:

    doc = nlp(text)
    claims = [
        sent.text.strip()
        for sent in doc.sents
        if len(sent.text.strip()) > 10
    ]
    return claims

In [ ]:
def build_search_query(claim: str) -> str:
    
    doc = nlp(claim)
    entities = [ent.text for ent in doc.ents]
    if entities:
        return " ".join(entities[:4])
    return claim[:60]


In [ ]:

def get_evidence(claim: str, top_k: int = 3) -> List[str]:
    
    query = build_search_query(claim)

    try:
        search_results = wikipedia.search(query, results=3)
        if not search_results:
            return []

        page_content = None
        for title in search_results:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                page_content = page.content
                break
            except (wikipedia.DisambiguationError, wikipedia.PageError):
                continue

        if not page_content:
            return []

        sentences = [
            s.strip()
            for s in page_content.split(".")
            if len(s.strip()) > 25
        ]
        if not sentences:
            return []

        claim_vec = sbert.encode(claim, convert_to_tensor=True)
        sent_vecs = sbert.encode(sentences, convert_to_tensor=True)
        scores    = util.cos_sim(claim_vec, sent_vecs)[0]

        top_indices = scores.topk(min(top_k, len(sentences))).indices.tolist()
        return [sentences[i] for i in top_indices]

    except Exception as e:
        print(f"  [evidence] Wikipedia lookup failed for '{query}': {e}")
        return []

In [12]:
evidence = get_evidence("Alexander Graham Bell invented the telephone in France")
for e in evidence:
    print(e)

• The apparatus used in sending the message • Was the Photophone invented by • Alexander Graham Bell • inventor of the telephone • This plaque was placed here by • Alexander Graham Bell Chapter • Telephone Pioneers of America
, honoring Bell's invention of the Photophone, the precursor of fibre-optical communications, and which he referred to as his 'greatest invention'
, Sculptor, was placed here through International subscription by the Bell Telephone Memorial Association to mark the invention of the Telephone at Brantford by Alexander Graham Bell in 1874"
